In [ ]:
import matplotlib.pyplot as plt 
import numpy as np

nb_agents = 1000
nb_secteurs = 10

A_brut = np.random.uniform(0, 1, (nb_agents, nb_secteurs))
A = A_brut / A_brut.sum(axis=1, keepdims=True)

In [ ]:
def Y(P, Pi):
    return P * Pi 

def J(Y_mat): 
    return np.argmax(Y_mat, axis=1)

def B(P, Pi):
    return np.max(Y(P, Pi), axis=1)

def Offre(P, Pi):
    Omega = np.zeros(nb_secteurs)
    Y_mat = Y(P, Pi)
    choix = J(Y_mat)
    for j in range(nb_secteurs):
        Omega[j] = np.sum(Y_mat[choix == j, j])
    return Omega

def Demande(A, B_vec):
    return B_vec @ A


def esperances(A, Pi,P_batch):
    Y_batch = P_batch * Pi

    #on utilise les fonctions numpy pr aller plus vite
    choix = np.argmax(Y_batch, axis=2)
    B_batch = np.max(Y_batch, axis=2)
    
    masque_choix = np.eye(nb_secteurs)[choix]
    Omega_batch = np.sum(Y_batch * masque_choix, axis=1)
    Demande_batch = B_batch @ A
    
    return np.mean(Omega_batch, axis=0), np.mean(Demande_batch, axis=0)

In [ ]:
def algo_naif(A, n_iter=200, learning_rate=0.01,n=1000):
    Pi = np.ones(nb_secteurs) / nb_secteurs
    historique_Pi = [Pi.copy()]
    P_batch = np.random.uniform(0, 1, (n, nb_agents, nb_secteurs))
    
    for i in range(n_iter):
        E_Offre, E_Demande = esperances(A, Pi, P_batch)
        Z = E_Demande - E_Offre 
        
        #méthode à pas
        Pi = Pi + learning_rate * Z
        
        Pi = np.maximum(Pi, 0.0001) #on veut pas pi=0
        Pi = Pi / np.sum(Pi) #normalise

        #méthode point fixe
        # Z_plus = np.maximum(0, Z)
        # Pi = (Pi + Z_plus) / (np.sum(Pi) + np.sum(Z_plus))
        
        historique_Pi.append(Pi.copy())
        erreur = np.max(np.abs(Z))
        print(i)
        if erreur < 0.05: 
            print(f"Équilibre atteint à l'itération {i}")
            break
            
    return Pi, np.array(historique_Pi)

Pi_eq, historique = algo_naif(A)

In [ ]:
# Tracer l'évolution des prix vers l'équilibre
plt.figure(figsize=(10, 6))
plt.plot(historique)
plt.title("Convergence des prix d'équilibre (Méthode du point fixe)")
plt.xlabel("Nombre d'itérations")
plt.ylabel("Prix sectoriels")
plt.grid(True)
plt.show()